# Lecture 10: Building Vanilla RAG with LCEL

**Course:** NLP with LangChain | **Platform:** Hope to Skill  
**Duration:** ~20 minutes | **Level:** Intermediate  

---

## The Big Picture

This is it — **your first complete RAG system!** Everything you've learned  
in the previous lectures comes together right here, right now.

> Think of LCEL (LangChain Expression Language) as **plumbing for AI**.  
> You connect pipes together: question flows in, retriever finds documents,  
> prompt formats everything, LLM generates the answer, clean text flows out.  
> The `|` pipe operator connects each stage — just like real plumbing!

### What You Will Learn

| # | Topic | Real-World Analogy |
|---|-------|--------------------|  
| 1 | What is LCEL? | Plumbing pipes connecting AI components |
| 2 | LCEL vs traditional chains | Modern plumbing vs bucket brigade |
| 3 | The 4 RAG components | Four pipes in your water system |
| 4 | Building the knowledge base | Filling the water tank |
| 5 | Prompt templates | The filter that shapes the output |
| 6 | Assembling the chain | Connecting all the pipes |
| 7 | Running & debugging | Turning on the faucet |
| 8 | Streaming responses | Water flowing continuously, not in buckets |

> **After this lecture:** You'll have a working end-to-end RAG system  
> in ~20 lines of code. By end of this lecture, Module 2 is complete!

---

## 0. Environment Setup

Run this cell **once** to install the packages we'll need today.

**Important:** This lecture requires an **OpenAI API key** because we need  
an LLM to generate answers. If you don't have one:
1. Go to [platform.openai.com](https://platform.openai.com/)
2. Create an account and add credits ($5 is plenty for this course)
3. Generate an API key under "API Keys"

In [2]:
# Install required packages (run once, then you can skip this cell)
# langchain-openai: OpenAI LLM integration (ChatGPT)
# langchain-qdrant: Qdrant vector store with LangChain support
# langchain-huggingface: free embedding models (same as Lecture 7)
%pip install langchain langchain-openai langchain-qdrant langchain-huggingface qdrant-client

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

# Set your OpenAI API key here
# Replace "your-api-key-here" with your actual key from platform.openai.com
# IMPORTANT: Never share your API key or commit it to git!
os.environ["OPENAI_API_KEY"] = ""

# Verify the key is set (we only show the first 8 characters for security)
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key and api_key != "your-api-key-here":
    # [:8] shows only the first 8 characters, hiding the rest for security
    print(f"API key set: {api_key[:8]}...")
else:
    print("WARNING: Please set your OpenAI API key above!")
    print("Replace 'your-api-key-here' with your actual key.")

API key set: sk-proj-...


---

## 1. What Is LCEL? (LangChain Expression Language)

**LCEL** is a **declarative way to compose AI pipelines** — think of it as  
connecting Lego blocks to build something bigger.

### The Magic: The Pipe Operator `|`

The `|` operator chains components together. The output of one step  
becomes the input of the next — just like Unix pipes:

```
Unix:    cat file.txt | grep "error" | sort   | head -5
LCEL:    retriever    | format_docs  | prompt | llm | parser
```

| Without LCEL | With LCEL |
|-------------|-----------|  
| Create each component separately | Same |
| Write glue code to connect them | Just use `\|` |
| Handle errors manually | Built-in error handling |
| Add streaming later | Streaming built-in |

> **Key insight:** Each `|` is one step. You can read the chain  
> left to right and understand exactly what happens at each stage.

Let's see it in action with a simple example first!

In [4]:
# Let's start simple — a chain WITHOUT RAG to understand the | operator

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Component 1: A prompt template with one variable {topic}
prompt = ChatPromptTemplate.from_template(
    "Tell me one interesting fact about {topic} in 2 sentences."
)

# Component 2: The LLM (ChatGPT)
# temperature=0 means deterministic (same answer every time) — great for facts
# gpt-4o-mini is fast and cheap — perfect for learning
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Component 3: Output parser — extracts just the text string from LLM response
parser = StrOutputParser()

# Connect them with | (the LCEL pipe operator!)
# Data flows left to right: {topic} -> prompt -> llm -> parser -> string
simple_chain = prompt | llm | parser

# .invoke() runs the chain with the given input
result = simple_chain.invoke({"topic": "natural language processing"})
print(result)

c:\Users\mhari\miniconda3\envs\demo1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Natural language processing (NLP) has advanced significantly due to the development of deep learning techniques, enabling machines to understand and generate human language with remarkable accuracy. One interesting fact is that models like OpenAI's GPT-3 can generate coherent and contextually relevant text, making them capable of tasks ranging from creative writing to coding assistance.


#### What just happened?

We built our first LCEL chain in 3 lines:

```python
simple_chain = prompt | llm | parser
```

Here's what happened when we called `.invoke()`:

1. **prompt** received `{"topic": "NLP"}` and produced a formatted message
2. **`|`** passed that message to the next component
3. **llm** received the message, sent it to OpenAI, got a response
4. **`|`** passed the response to the next component
5. **parser** received the AI response and extracted just the text string

**That's the beauty of LCEL:** each `|` is a pipe, and data flows through  
from left to right. No glue code, no boilerplate. Clean and readable.

---

## 2. LCEL vs Traditional Chains

Before LCEL, building chains in LangChain was verbose and hard to debug.  
LCEL makes everything cleaner:

| Feature | Traditional Chains | LCEL |
|---------|-------------------|------|
| **Readability** | Nested classes, hard to follow | Read left to right with `\|` |
| **Swapping components** | Rewrite multiple lines | Change one component |
| **Streaming** | Manual implementation | Built-in with `.stream()` |
| **Async support** | Manual with asyncio | Built-in with `.ainvoke()` |
| **Error handling** | Try/except everywhere | Built-in fallbacks |
| **Debugging** | Print statements | Inspect each step separately |

### Example: Traditional vs LCEL

```python
# Traditional (verbose — old approach)
from langchain.chains import LLMChain
chain = LLMChain(llm=llm, prompt=prompt, output_parser=parser)
result = chain.run(topic="NLP")

# LCEL (clean — modern approach!)
chain = prompt | llm | parser
result = chain.invoke({"topic": "NLP"})
```

> **Rule of thumb:** If you see `LLMChain`, `RetrievalQA`, or other legacy  
> chain classes in tutorials, they're using the old approach. Always prefer LCEL.

---

## 3. The 4 LCEL Components for RAG

To build a RAG system, we need exactly **4 components** connected with `|`:

```
question --> [Retriever] --> [Prompt] --> [LLM] --> [Parser] --> answer
                |              |           |          |
          finds relevant   formats      generates   extracts
           documents      context +    the answer    plain
                         question                    text
```

| # | Component | What It Does | LangChain Class |
|---|-----------|-------------|-----------------|  
| 1 | **Retriever** | Question -> list of relevant docs | `vectorstore.as_retriever()` |
| 2 | **Prompt Template** | Formats context + question for LLM | `ChatPromptTemplate` |
| 3 | **LLM** | Generates answer from formatted prompt | `ChatOpenAI` |
| 4 | **Output Parser** | Extracts just the text string | `StrOutputParser` |

Each component is a **separate, swappable module**:
- Want to use a different LLM? Change one line
- Want to use a different vector store? Change one line
- Want a different prompt? Change one line

> **This is the power of LCEL** — each pipe section is independent.  
> You can swap any component without touching the rest of the chain.

---

## 4. Building the Knowledge Base

Before we can build the RAG chain, we need a searchable knowledge base.  
This combines everything from Lectures 5-8:

```
Load (L5) --> Split (L6) --> Embed (L7) --> Store (L8) --> Ready for RAG!
```

We'll use our familiar `nlp_article.txt` and Qdrant Cloud.

In [ ]:
# ============================================================
# BUILD THE KNOWLEDGE BASE (Lectures 5-8 in one cell!)
# ============================================================

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

# ============================================================
# QDRANT CLOUD SETUP
# ============================================================
# Use the same credentials from Lectures 8 & 9
# Replace the placeholders with your actual Qdrant Cloud credentials
# ============================================================

QDRANT_URL = ""
QDRANT_API_KEY = ""

# --- Step 1: LOAD (Lecture 5) ---
loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
documents = loader.load()
print(f"Step 1 - Loaded: {len(documents)} document(s)")

# --- Step 2: SPLIT (Lecture 6) ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # ~500 characters per chunk
    chunk_overlap=50,   # 50-char overlap so context isn't lost at boundaries
)
chunks = splitter.split_documents(documents)
print(f"Step 2 - Split into: {len(chunks)} chunks")

# --- Step 3 & 4: EMBED + STORE (Lectures 7 & 8) ---
# HuggingFaceEmbeddings wraps sentence-transformers for LangChain
# Same model as Lecture 7, but now in LangChain-compatible format
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# QdrantVectorStore.from_documents() does embedding + storage in one call:
# - Embeds all chunks automatically using the embeddings model
# - Stores them in Qdrant Cloud with their metadata
vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="nlp_course",  # Name of the collection
)

print(f"Step 3&4 - Embedded and stored {len(chunks)} chunks in Qdrant Cloud")
print(f"         Visit your dashboard to see the 'nlp_course' collection!")
print(f"\nKnowledge base ready!")

Step 1 - Loaded: 1 document(s)
Step 2 - Split into: 21 chunks


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 426.35it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step 3&4 - Embedded and stored 21 chunks in Qdrant Cloud
         Visit your dashboard to see the 'nlp_course' collection!

Knowledge base ready!


In [ ]:
# ============================================================
# BUILD THE KNOWLEDGE BASE — OpenAI Embeddings Version
# ============================================================
# EMBEDDING METHOD: OpenAI Embeddings (PAID, requires API key)
#
# This cell uses OpenAI embeddings (requires API key and costs money)
# For FREE local embeddings, use the previous cell instead
#
# Same pipeline, but with OpenAI's text-embedding-3-small model
# ============================================================

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

# ============================================================
# OPENAI API KEY SETUP
# ============================================================
import os
os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # Replace with your key
# ============================================================

# ============================================================
# QDRANT CLOUD SETUP
# ============================================================
QDRANT_URL = ""
QDRANT_API_KEY = ""

print("=" * 70)
print("KNOWLEDGE BASE SETUP - OpenAI Embeddings Version")
print("=" * 70)

# --- Step 1: LOAD (Lecture 5) ---
loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
documents = loader.load()
print(f"[SUCCESS] Step 1 - Loaded: {len(documents)} document(s)")

# --- Step 2: SPLIT (Lecture 6) ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # ~500 characters per chunk
    chunk_overlap=50,   # 50-char overlap so context isn't lost at boundaries
)
chunks = splitter.split_documents(documents)
print(f"[SUCCESS] Step 2 - Split into: {len(chunks)} chunks")

# --- Step 3 & 4: EMBED + STORE (Lectures 7 & 8) ---
# OpenAI embeddings - 1536 dimensions, requires API key
print(f"[PROGRESS] Step 3 - Initializing OpenAI embeddings...")
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")
print(f"[SUCCESS] OpenAI embedding model initialized (1536 dimensions)")

# QdrantVectorStore.from_documents() embeds and stores in one call
# Note the different collection name to avoid conflicts with free version
print(f"[PROGRESS] Step 4 - Embedding and storing in Qdrant Cloud...")
vectorstore_openai = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings_openai,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,  
    collection_name="nlp_course_openai",  # Different name from free version
)

print(f"[SUCCESS] Embedded and stored {len(chunks)} chunks in Qdrant Cloud")
print(f"  Collection name: nlp_course_openai")
print(f"  Embedding dimensions: 1536")
print(f"  Model: text-embedding-3-small")
print(f"  Estimated cost: ~$0.02 per 1M tokens")
print(f"\n" + "=" * 70)
print("KNOWLEDGE BASE READY - OpenAI Version")
print("=" * 70)
print(f"\n[INFO] Comparison:")
print(f"  - FREE (SentenceTransformer): 384 dims, runs locally, $0 cost")
print(f"  - PAID (OpenAI): 1536 dims, API-based, ~$0.02/1M tokens")
print(f"\n[INFO] Visit your Qdrant dashboard to see both collections!")
print(f"  - nlp_course (free)")
print(f"  - nlp_course_openai (paid)")

#### What just happened?

We built the entire knowledge base in one cell — combining 4 lectures:

| Step | Lecture | Code | What It Did |
|------|---------|------|-------------|  
| Load | L5 | `TextLoader("data/nlp_article.txt")` | Read the file |
| Split | L6 | `RecursiveCharacterTextSplitter(500, 50)` | Break into chunks |
| Embed | L7 | `HuggingFaceEmbeddings("all-MiniLM-L6-v2")` | Convert to vectors |
| Store | L8 | `QdrantVectorStore.from_documents()` | Store in Qdrant Cloud |

**New here:** `HuggingFaceEmbeddings` wraps our familiar `all-MiniLM-L6-v2`  
model in a LangChain-compatible interface. Same model, same 384 dimensions —  
just a wrapper so LangChain's vector store can use it directly.

**Also new:** `QdrantVectorStore.from_documents()` does embedding + storage  
in a single call. No manual `model.encode()` or `PointStruct` needed!

The chunks are now in **Qdrant Cloud** — check your dashboard!

---

## 5. Building the Prompt Template

The prompt template is where you **control the LLM's behavior**.  
A good RAG prompt tells the LLM:
- Here's the context (retrieved documents)
- Here's the question
- Stay grounded — only answer from the context

### The RAG Prompt Formula

```
"Based on the following context, answer the question.
 If the answer is not in the context, say so.

 Context: {context}

 Question: {question}"
```

The two variables `{context}` and `{question}` will be filled  
automatically by the chain at runtime.

In [6]:
# Build the RAG prompt template

from langchain_core.prompts import ChatPromptTemplate

# The template has 2 variables:
# {context} — will be filled with retrieved document chunks
# {question} — will be filled with the user's question
rag_prompt = ChatPromptTemplate.from_template(
    """You are a helpful teaching assistant for an NLP course.
Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:"""
)

# Let's inspect the template
print("Prompt Template Variables:")
# .input_variables shows what variables the template expects
print(f"  Required inputs: {rag_prompt.input_variables}")

# Preview: what the prompt looks like with sample values
sample = rag_prompt.format(
    context="NLP is a branch of AI that focuses on human language...",
    question="What is NLP?"
)
print(f"\nSample formatted prompt:")
print(f"{'=' * 50}")
# [:300] shows first 300 chars of the formatted prompt
# Remove [:300] to see the entire prompt
print(sample[:300])

Prompt Template Variables:
  Required inputs: ['context', 'question']

Sample formatted prompt:
Human: You are a helpful teaching assistant for an NLP course.
Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
NLP is a branch of AI that focuses on human language...

Question: What is NLP?


#### What just happened?

- We created a **prompt template** with two placeholders: `{context}` and `{question}`
- The prompt tells the LLM to **only answer from the provided context** —  
  this prevents hallucination (making up facts)
- `"I don't have enough information"` is our safety net — if the context  
  doesn't contain the answer, the LLM should say so honestly
- `.input_variables` shows what the template expects: `['context', 'question']`

**Why is this important?** The prompt is where you control the LLM's personality  
and behavior. Want shorter answers? Add "Be concise." Want bullet points?  
Add "Format as a bullet list." The prompt is your steering wheel.

---

## 6. Assembling the RAG Chain — The Main Event!

Now we connect all 4 components with the `|` pipe operator.  
This is where everything comes together — the moment you've been  
building toward since Lecture 5!

### The Chain Architecture

```
Question (string)
    |
    v
{"context": retriever | format_docs, "question": passthrough}
    |
    v
rag_prompt (fills {context} and {question})
    |
    v
llm (generates answer)
    |
    v
StrOutputParser() (extracts text)
    |
    v
Answer (string)
```

In [7]:
# ============================================================
# ASSEMBLING THE RAG CHAIN
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- Component 1: RETRIEVER ---
# .as_retriever() converts the vector store into a retriever
# search_kwargs={"k": 5} means "return the 5 most similar chunks"
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


# --- Helper: FORMAT DOCS ---
# The retriever returns Document objects, but the prompt needs a string
# This function joins all retrieved chunks into one text block
def format_docs(docs):
    """Join retrieved document chunks into a single context string."""
    # "\n\n" puts a blank line between each chunk for readability
    # doc.page_content gets the text from each Document object
    return "\n\n".join(doc.page_content for doc in docs)


# --- Component 2: PROMPT (created in Section 5 above) ---
# rag_prompt is our ChatPromptTemplate with {context} and {question}

# --- Component 3: LLM ---
# temperature=0 means deterministic — same question always gets same answer
# This is important for factual RAG (no creative variation)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- Component 4: OUTPUT PARSER ---
parser = StrOutputParser()

# ============================================================
# THE CHAIN — connect everything with |
# ============================================================
rag_chain = (
    {
        # "context": retriever finds docs, then format_docs joins them
        "context": retriever | format_docs,
        # "question": RunnablePassthrough() passes the question unchanged
        "question": RunnablePassthrough(),
    }
    | rag_prompt   # Fills {context} and {question} into the template
    | llm          # Sends the formatted prompt to ChatGPT
    | parser        # Extracts just the text string from the response
)

print("RAG chain assembled!")
print(f"\nChain components:")
print(f"  1. Retriever: Qdrant vector store (top 5 chunks)")
print(f"  2. Prompt: RAG template with context + question")
print(f"  3. LLM: gpt-4o-mini (temperature=0)")
print(f"  4. Parser: StrOutputParser (plain text output)")

RAG chain assembled!

Chain components:
  1. Retriever: Qdrant vector store (top 5 chunks)
  2. Prompt: RAG template with context + question
  3. LLM: gpt-4o-mini (temperature=0)
  4. Parser: StrOutputParser (plain text output)


#### What just happened?

We assembled the complete RAG chain in **one expression**:

```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | parser
)
```

Let's break down each piece:

| Piece | What It Does |
|-------|-------------|
| `retriever \| format_docs` | Finds 5 relevant chunks, joins them into one string |
| `RunnablePassthrough()` | Passes the user's question through unchanged |
| `{...}` dict | Creates a dict with `"context"` and `"question"` keys |
| `\| rag_prompt` | Fills those values into the template |
| `\| llm` | Sends the formatted prompt to ChatGPT |
| `\| parser` | Extracts the answer as a plain string |

**`RunnablePassthrough()`** is the key to understanding the first step:  
it takes the user's question string and passes it through as-is, while  
simultaneously the retriever uses that same question to find relevant docs.

---

## 7. Running the Chain + Inspecting Output

Time to test our RAG system! We'll ask questions about our NLP article  
and see if the answers are grounded in the actual document content.

In [8]:
# Let's ask our first question!

question = "What is Natural Language Processing?"

# .invoke() runs the entire chain:
# question -> retriever -> format_docs -> prompt -> LLM -> parser -> answer
answer = rag_chain.invoke(question)

print(f"Question: {question}")
print(f"{'=' * 60}")
print(f"\nAnswer:\n{answer}")

Question: What is Natural Language Processing?

Answer:
Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a manner that is both valuable and meaningful.


#### What just happened?

We ran the complete RAG pipeline with one line: `rag_chain.invoke(question)`

Behind the scenes, these steps happened automatically:
1. The **retriever** embedded the question and searched Qdrant for similar chunks
2. **format_docs** joined the retrieved chunks into a context string
3. The **prompt** formatted the context + question into a message
4. The **LLM** (ChatGPT) generated an answer grounded in the context
5. The **parser** extracted the answer as a plain string

**The answer should cite facts from the NLP article** — not make up information.  
That's the power of RAG: grounded, factual answers from your own documents!

In [10]:
# Let's INSPECT what the retriever finds — this is how you debug RAG

question = "what is hope to skill ?"

# Call the retriever DIRECTLY to see what chunks it found
# This is separate from the chain — just for debugging
retrieved_docs = retriever.invoke(question)

print(f"Question: '{question}'")
print(f"\n--- Retrieved {len(retrieved_docs)} chunks ---")

# This loop shows each retrieved chunk
# enumerate() gives us the index (i=0,1,2...) and the Document object
for i, doc in enumerate(retrieved_docs):
    print(f"\n  Chunk #{i + 1}:")
    # [:150] shows first 150 characters of the chunk
    # To see the full chunk text, remove [:150]
    print(f"  {doc.page_content[:150]}...")

# Now run the full chain to see the final answer
print(f"\n{'=' * 60}")
print(f"FULL CHAIN ANSWER:")
answer = rag_chain.invoke(question)
print(answer)

Question: 'what is hope to skill ?'

--- Retrieved 5 chunks ---

  Chunk #1:
  GPT (Generative Pre-trained Transformer) models, developed by OpenAI, take a different approach. They are trained to predict the next word in a sequen...

  Chunk #2:
  Retrieval-Augmented Generation (RAG) combines retrieval and generation to produce accurate, grounded responses. Instead of relying solely on the LLM's...

  Chunk #3:
  Evaluation metrics matter. Different NLP tasks require different evaluation metrics. For classification tasks, use accuracy, precision, recall, and F1...

  Chunk #4:
  Monitor and maintain. NLP models can degrade over time as language evolves and data distributions shift. Implement monitoring to track model performan...

  Chunk #5:
  The transformer architecture, introduced in the landmark paper "Attention Is All You Need" by Vaswani et al. in 2017, revolutionized NLP. Unlike previ...

FULL CHAIN ANSWER:
I don't have enough information to answer that.


#### What just happened?

We did two things:
1. **Called the retriever directly** with `retriever.invoke()` to see which chunks it found
2. **Ran the full chain** to see the final answer

This is how you **debug a RAG system**:
- If the answer is wrong, check the retrieved chunks first
- If the retrieved chunks are irrelevant, the problem is in the retriever  
  (try different chunk sizes, overlap, or embedding models)
- If the chunks are relevant but the answer is wrong, the problem is in the  
  prompt (try being more specific about how to use the context)

**Pro tip:** Always log the retrieved documents during development.  
It's the fastest way to find and fix RAG issues.

In [11]:
# Let's test with 3 different questions to evaluate our RAG

questions = [
    "What are the main NLP tasks?",
    "Who created the transformer architecture and when?",
    "What is RAG and how does it work?",
]

# This loop asks each question and prints the answer
# enumerate() gives us the index (i=0,1,2) and the question string
for i, question in enumerate(questions):
    print(f"\nQ{i + 1}: {question}")
    print(f"{'-' * 50}")

    # .invoke() runs the full chain for each question
    answer = rag_chain.invoke(question)
    print(f"A: {answer}")


Q1: What are the main NLP tasks?
--------------------------------------------------
A: I don't have enough information to answer that.

Q2: Who created the transformer architecture and when?
--------------------------------------------------
A: The transformer architecture was introduced by Vaswani et al. in 2017.

Q3: What is RAG and how does it work?
--------------------------------------------------
A: Retrieval-Augmented Generation (RAG) combines retrieval and generation to produce accurate, grounded responses. It works by first retrieving relevant documents from a knowledge base and then using those documents as context for the LLM to generate an answer. This approach reduces hallucination and allows the system to work with up-to-date information.


#### What just happened?

We asked 3 different questions and got answers grounded in our NLP article:

- **"What are the main NLP tasks?"** — Should mention text classification,  
  NER, sentiment analysis, machine translation, question answering
- **"Who created the transformer architecture?"** — Should mention  
  Vaswani et al., "Attention Is All You Need," 2017
- **"What is RAG?"** — Should describe retrieval-augmented generation  
  from the LangChain chapter

**Check:** Are the answers consistent with what's in `nlp_article.txt`?  
If yes, your RAG system is working correctly! The LLM is using the  
retrieved context, not making things up from its training data.

---

## 8. Streaming Responses — Better User Experience

### The Problem

Without streaming, users wait 5-10 seconds staring at a blank screen  
until the entire answer is generated. This feels painfully slow.

### The Solution

With streaming, users see the answer **forming word by word** as the LLM  
generates it — like watching someone type. Same total time, but it  
*feels* much faster and more responsive.

| Without Streaming | With Streaming |
|------------------|----------------|
| Wait 5 seconds... then full answer | First word in 0.5 sec, flows continuously |
| User thinks "is it broken?" | User reads as answer forms |
| Bad for chat interfaces | Essential for chat interfaces |

> **Key insight:** Streaming doesn't make the LLM faster — it just shows  
> tokens as they're generated instead of waiting for the complete response.

In [13]:
# Streaming: see the answer form word by word!

question = "Explain how BERT works in simple terms."

print(f"Question: {question}")
print(f"{'=' * 60}")
print(f"Streaming answer:\n")

# .stream() returns an iterator of text chunks
# Each chunk is a small piece of the answer (usually 1-3 words)
for chunk in rag_chain.stream(question):
    # end="" prevents adding a newline after each chunk
    # flush=True forces the output to display immediately
    print(chunk, end="", flush=True)

# Add a newline at the end so the next print starts on a new line
print("\n")
print(f"{'=' * 60}")
print("The answer streamed token by token — much better UX!")

Question: Explain how BERT works in simple terms.
Streaming answer:

BERT works by reading text in both directions, looking at the words that come before and after each word. This helps it understand the full context and meaning of the words better than models that only read in one direction. It is trained on a lot of text data first and then fine-tuned for specific tasks, making it very effective for understanding language.

The answer streamed token by token — much better UX!


#### What just happened?

- `.stream()` replaces `.invoke()` — same chain, different output mode
- Instead of returning the full answer at once, it yields **small chunks**  
  (tokens) as the LLM generates them
- `end=""` prevents Python from adding a newline after each chunk
- `flush=True` forces the display to update immediately

**When to use which:**

| Method | Use When |
|--------|----------|
| `.invoke()` | You need the full answer as a variable (for processing) |
| `.stream()` | Displaying to a user in real-time (chat interfaces) |
| `.ainvoke()` | Async version of invoke (for web servers) |
| `.astream()` | Async version of stream (for web servers) |

**Streaming is essential for any real chat application.** Users expect to see  
the response forming — waiting for the full answer is not acceptable UX.

---

## 9. Hands-On: Complete RAG with Logging & Timing

Let's build a polished RAG function that includes:
- Answer generation
- Retrieval logging (see which chunks were used)
- End-to-end latency timing
- Clean formatted output

In [14]:
import time


def ask_rag(question, chain=rag_chain, ret=retriever, show_sources=True):
    """Ask a question and get a RAG-powered answer with source logging."""

    print(f"\nQuestion: {question}")
    print(f"{'=' * 65}")

    # Time the entire RAG pipeline
    start_time = time.time()

    # Run the chain to get the answer
    answer = chain.invoke(question)

    # Calculate how long it took
    # (time.time() - start_time) gives seconds, * 1000 converts to ms
    elapsed_ms = (time.time() - start_time) * 1000

    print(f"\nAnswer:\n{answer}")
    # :.0f formats the number with zero decimal places
    print(f"\n--- Latency: {elapsed_ms:.0f} ms ---")

    # Optionally show which chunks were retrieved
    if show_sources:
        retrieved_docs = ret.invoke(question)
        print(f"\n--- Sources ({len(retrieved_docs)} chunks retrieved) ---")
        # This loop shows a preview of each source chunk
        for i, doc in enumerate(retrieved_docs):
            # [:100] shows first 100 chars; remove to see full chunk
            print(f"  [{i + 1}] {doc.page_content[:100]}...")

    return answer


# Test with 5 different questions!
questions = [
    "What is Natural Language Processing?",
    "What are the main NLP tasks described in the article?",
    "How do transformers work?",
    "What is LangChain used for?",
    "What are best practices for NLP projects?",
]

# This loop asks each question using our ask_rag function
for question in questions:
    ask_rag(question)
    print()


Question: What is Natural Language Processing?

Answer:
Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a manner that is both valuable and meaningful.

--- Latency: 4146 ms ---

--- Sources (5 chunks retrieved) ---
  [1] Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?
...
  [2] NLP combines computational linguistics — rule-based modeling of human language — with statistical, m...
  [3] Chapter 2: Core NLP Tasks

Text Classification is one of the most fundamental tasks in NLP. It invol...
  [4] The transformer architecture, introduced in the landmark paper "Attention Is All You Need" by Vaswan...
  [5] The history of NLP dates back to the 1950s when Alan Turing published his famous article "Computing ...


Qu

#### What just happened?

We built a production-style RAG function with:

1. **Timing** — shows how long each question takes (typically 1-5 seconds)
2. **Source logging** — shows which chunks the retriever found
3. **Clean output** — formatted for easy reading and evaluation

**How to evaluate your RAG answers:**

| Check | Good Sign | Bad Sign |
|-------|-----------|----------|
| Factual accuracy | Answer matches the source document | Answer contains made-up facts |
| Source relevance | Retrieved chunks relate to the question | Retrieved chunks are random |
| Groundedness | Answer only uses info from context | Answer adds info not in context |
| Completeness | Answer covers the key points | Answer is too vague |

If you notice bad answers, the fix is usually:
- **Wrong chunks retrieved?** Adjust chunk_size, overlap, or k
- **Right chunks, wrong answer?** Adjust the prompt template
- **Too slow?** Use a smaller/faster model (gpt-4o-mini is fastest)

---

## 10. Mini Challenges

### Challenge 1: Custom Prompt Engineering
Modify the `rag_prompt` template to make the LLM answer in **bullet points**  
and keep answers under 3 sentences. Rebuild the chain and test it.

### Challenge 2: Your Own Knowledge Base RAG
Load a different document (a text file, PDF, or web page from Lecture 5),  
build a complete RAG system with LCEL, and ask 3 questions about it.

### Challenge 3: Temperature Experiment
Create two chains — one with `temperature=0` and one with `temperature=0.9`.  
Ask the same question 3 times on each chain. How do the answers differ?  
Which is better for factual RAG?

> **Hint for Challenge 3:** `temperature=0` should give consistent, factual  
> answers. `temperature=0.9` will be more creative but may hallucinate.

In [ ]:
# ============================================================
# Challenge 1: Custom Prompt Engineering
# ============================================================
# Modify rag_prompt to output bullet points, max 3 sentences
# Rebuild the chain and test with a question

# Your code here:


In [ ]:
# ============================================================
# Challenge 2: Your Own Knowledge Base RAG
# ============================================================
# 1. Load a different document (text, PDF, or web page)
# 2. Split -> Embed -> Store in Qdrant
# 3. Build an LCEL chain
# 4. Ask 3 questions

# Your code here:


In [ ]:
# ============================================================
# Challenge 3: Temperature Experiment
# ============================================================
# Create two chains: temperature=0 and temperature=0.9
# Ask the same question 3 times on each
# Compare: which gives more consistent, factual answers?

# Your code here:


---

## 11. Quick Reference: LCEL RAG Cheat Sheet

### The Complete RAG Chain Pattern

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 2. Format helper
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Prompt
prompt = ChatPromptTemplate.from_template(
    "Context: {context}\n\nQuestion: {question}\n\nAnswer:"
)

# 4. LLM + Parser
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 5. Chain
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

# Run
answer = chain.invoke("Your question here")

# Stream
for chunk in chain.stream("Your question here"):
    print(chunk, end="", flush=True)
```

### Key Settings

| Setting | Value | Why |
|---------|-------|-----|
| `temperature` | `0` | Factual, consistent answers (no creativity) |
| `k` | `5` | Number of chunks to retrieve (3-10 is typical) |
| `model` | `"gpt-4o-mini"` | Fast and cheap. Use `"gpt-4o"` for best quality |
| `chunk_size` | `500` | Good balance for most documents |
| `chunk_overlap` | `50` | Prevents losing context at boundaries |

### The Golden Rules

> **`temperature=0` for RAG.** You want factual, grounded answers —  
> not creative storytelling. Save `temperature > 0` for creative tasks.

> **Always embed queries with the SAME model** you used for documents.  
> Mixing models gives meaningless results (from Lecture 7).

---

## 12. Key Takeaways — Module 2 Complete!

1. **LCEL** = LangChain Expression Language — connect components with `|`
2. **4 RAG components:** retriever | format_docs | prompt | LLM | parser
3. **`temperature=0`** for factual, consistent RAG answers
4. **Streaming** (`.stream()`) is essential for real user-facing applications
5. **Debug by inspecting retrieved chunks** — most RAG problems are retrieval problems
6. **The complete RAG pipeline:**

```
Load (L5) --> Split (L6) --> Embed (L7) --> Store (L8) --> RAG Chain (L10)
```

### You Now Have a Working RAG System!

**Module 2 is complete.** You can now:
- Load documents from any source
- Split them intelligently into chunks
- Embed them into meaningful vectors
- Store them in a searchable vector database
- Build an LCEL chain that answers questions from your documents
- Stream responses for great user experience

### What's Next: Module 3

**Make it smarter, faster, and more accurate:**
- Advanced retrieval strategies (MMR, multi-query)
- Evaluation and testing
- Production deployment

---

*Hope to Skill — Building the future, one skill at a time.*

---

## Appendix: PEP 8 Style Rules Used in This Notebook

All code in this notebook follows Python's PEP 8 style guide.  
Here are the rules applied, with examples from the code above.

### Naming Conventions

| Rule | Convention | Example from This Notebook |
|------|-----------|---------------------------|
| Variables & functions | `snake_case` | `rag_chain`, `format_docs()`, `ask_rag()` |
| Constants | `UPPER_CASE` | `OPENAI_API_KEY` (environment variable) |
| Classes | `PascalCase` | `ChatOpenAI`, `ChatPromptTemplate`, `StrOutputParser` (from libraries) |

### Import Rules

| Rule ID | Rule | Example |
|---------|------|---------|  
| E401 | One import per line | `from langchain_openai import ChatOpenAI` |
| E402 | Imports at the top of each section | All imports at cell top |
| — | Group: stdlib then third-party then local | `os`, `time` then `langchain` |

### Whitespace & Formatting

| Rule ID | Rule | Example |
|---------|------|---------|  
| E225 | Spaces around operators | `i + 1`, `elapsed_ms > 0` |
| E231 | Space after commas | `ChatOpenAI(model="gpt-4o-mini", temperature=0)` |
| E265 | Block comments start with `# ` | `# Run the chain to get the answer` |
| E303 | Two blank lines before function defs | See `format_docs()`, `ask_rag()` |
| E501 | Max line length of 79 characters | Long strings use triple-quote wrapping |

### Best Practices Used

| Practice | Why | Example |
|----------|-----|---------|  
| f-strings | Clean string formatting | `f"Latency: {elapsed_ms:.0f} ms"` |
| `enumerate()` | Index + value in loops | `for i, doc in enumerate(retrieved_docs)` |
| Generator expressions | Memory-efficient joins | `"\n\n".join(doc.page_content for doc in docs)` |
| `:.0f` formatting | Control decimal places | `elapsed_ms:.0f` |
| Docstrings | Explain function purpose | `format_docs()` and `ask_rag()` have docstrings |
| Descriptive names | Code reads like English | `rag_chain` not `rc`, `format_docs` not `fd` |
| Default parameters | Flexible function calls | `ask_rag(question, show_sources=True)` |
| Triple-quoted strings | Multi-line prompt templates | See `rag_prompt` template |

### Quick PEP 8 Cheat Sheet

```python
# GOOD (PEP 8 compliant)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | parser
)
answer = rag_chain.invoke("What is NLP?")
for i, doc in enumerate(retrieved_docs):
    print(f"Chunk #{i + 1}: {doc.page_content[:100]}")

# BAD (violates PEP 8)
rc = ({"context":retriever|format_docs,"question":RunnablePassthrough()}|rag_prompt|llm|parser)
a = rc.invoke("What is NLP?")                  # unclear variable names
for i,d in enumerate(retrieved_docs):           # single-letter vars, no space after comma
    print("Chunk #"+str(i+1)+": "+d.page_content[:100])  # string concat
```